# Optimizing Neural Networks
This notebook contains code to optimize a neural network, focusing on the number of neurons in the model and the learning rate of the model. The model is designed to take in genetic data from viral samples in the form of a `.fasta` file, and predict the probability of the sample originating from one of 3 given places. 

In [ ]:
from Bio import SeqIO
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import random
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from itertools import cycle
from temperature_scaling import ModelWithTemperature
import optuna
from sklearn.model_selection import KFold


alignment = list(SeqIO.parse('../../simulations/results/simulated_alignment_much_larger.fasta', 'fasta'))
metadata = pd.read_csv('../../simulations/results/simulated_metadata_much_larger.csv')

In [2]:
#Set to ensure reproducability
torch.manual_seed(77)
torch.use_deterministic_algorithms(True)
np.random.seed(77)

## Data Wrangling
In order for the model to be able to read the sequence data in the `.fasta` file, we must convert the strings of nucleotides into 1s and 0s through **one-hot encoding**. Additionally, we must specify the data's labels and features; split the data into training, test, and validation sets; and make the data compatible with PyTorch.

In [3]:
#Encodes one sequence
def oneHotEncode(seq): 
    seq2=list()
    mapping = {"A":[1.0, 0.0, 0.0, 0.], "T": [0.0, 1.0, 0.0, 0.0], "C": [0.0, 0.0, 1.0, 0.0], "G":[0.0, 0.0, 0.0, 1.0]}
    for i in seq:
        seq2.append(mapping[i] if i in mapping.keys() else [0., 0., 0., 0.])
    return np.array(seq2).flatten()

#Iterating encode function across an entire dataset
def encodeSet(dataList):
    set2 = list()
    for i in range(len(dataList)):
        set2.append(oneHotEncode(dataList[i]))
    return np.array(set2)

In [4]:
#Encode the given dataset
sequenceEncoded = pd.DataFrame(encodeSet(alignment))
allData = pd.concat([sequenceEncoded, metadata['subgroup']], axis = 1)

In [5]:
#Splitting data into features (X) and label (Y, subgroups of A, B, C)
X = allData.drop('subgroup', axis = 1).values.astype(np.float32)
y = allData['subgroup'].values

#Transform the categorical subgroup data (A, B, C) into numerical (0, 1, 2), similar to one-hot encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y).astype(np.int64)

In [6]:
#Split data into test and train groups
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, train_size=0.8, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1) # 0.25 x 0.8 = 0.2

In [7]:
#Creating a PyTorch Dataset
#PyTorch can only read the data when it has been transformed into a tensor
class LocationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = LocationDataset(X_train, y_train)
val_dataset = LocationDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

## The Model

In [8]:
#Defining the model. Here, the number of neurons per layer is left as a variable to be optimized.
class Net(nn.Module):
    def __init__(self, l1, l2, l3):
        super(Net, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(len(sequenceEncoded.columns), l1),
            nn.ReLU(),
            nn.Linear(l1, l2),
            nn.ReLU(),
            nn.Linear(l2, l3),
            nn.ReLU(),
            nn.Linear(l3, 3)
        )
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        logits = self.model(x)
        return logits / self.temperature

In [9]:
def objective(trial, train_loader, val_loader):
    l1 = trial.suggest_int('l1', 1, 512)
    l2 = trial.suggest_int('l2', 1, 512)
    l3 = trial.suggest_int('l3', 1, 512)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-1, log=True)

    model = Net(l1, l2, l3)
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(10):
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

        #Validation of the model
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(val_loader):
                output = model(data)
                pred = output.argmax(dim = 1, keepdim = True)
                correct += pred.eq(target.view_as(pred)).sum().item()
                total += 1

        accuracy = correct / total

    return accuracy

In [10]:
#Defining the objective of the optimization -- to get the model's accuracy as high as possible
def objective_cv(trial):
    accuracy = objective(trial, train_loader, val_loader)
    return accuracy

In [11]:
sampler = optuna.samplers.RandomSampler(seed = 77)
study = optuna.create_study(direction = "maximize", sampler = sampler)
study.optimize(objective_cv, n_trials = 15)

[I 2025-08-15 15:47:39,897] A new study created in memory with name: no-name-9c58272b-7b9a-42c7-8706-5654132b9ceb
[I 2025-08-15 15:48:21,430] Trial 0 finished with value: 22.69655172413793 and parameters: {'l1': 471, 'l2': 329, 'l3': 386, 'lr': 0.00026178436466168267}. Best is trial 0 with value: 22.69655172413793.
[I 2025-08-15 15:48:28,529] Trial 1 finished with value: 10.606896551724137 and parameters: {'l1': 45, 'l2': 404, 'l3': 167, 'lr': 0.004199556846960137}. Best is trial 0 with value: 22.69655172413793.
[I 2025-08-15 15:48:41,848] Trial 2 finished with value: 10.627586206896552 and parameters: {'l1': 124, 'l2': 280, 'l3': 206, 'lr': 0.013982204890346961}. Best is trial 0 with value: 22.69655172413793.
[I 2025-08-15 15:49:19,037] Trial 3 finished with value: 22.06206896551724 and parameters: {'l1': 429, 'l2': 302, 'l3': 152, 'lr': 0.000696711648959223}. Best is trial 0 with value: 22.69655172413793.
[I 2025-08-15 15:49:51,085] Trial 4 finished with value: 10.606896551724137 and

In [12]:
print("Best Hyperparameters:", study.best_params)

Best Hyperparameters: {'l1': 471, 'l2': 329, 'l3': 386, 'lr': 0.00026178436466168267}
